<a href="https://colab.research.google.com/github/AlinaSabitova/-_-/blob/main/lab4_etl.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#### Подготовка окружения

In [ ]:
!pip install "dask[complete]" graphviz

In [ ]:
import dask.dataframe as dd
import dask.delayed as delayed
from dask.distributed import Client
from dask.diagnostics import ProgressBar

# Инициализация клиента Dask (Оптимизированные настройки без жесткого лимита памяти)
client = Client(n_workers=2, threads_per_worker=2, processes=True)
client

INFO:distributed.http.proxy:To route to workers diagnostics web server please install jupyter-server-proxy: python -m pip install jupyter-server-proxy
INFO:distributed.scheduler:State start
INFO:distributed.scheduler:  Scheduler at:     tcp://127.0.0.1:36901
INFO:distributed.scheduler:  dashboard at:  http://127.0.0.1:8787/status
INFO:distributed.scheduler:Registering Worker plugin shuffle
INFO:distributed.nanny:        Start Nanny at: 'tcp://127.0.0.1:33541'
INFO:distributed.nanny:        Start Nanny at: 'tcp://127.0.0.1:46669'
INFO:distributed.scheduler:Register worker addr: tcp://127.0.0.1:42763 name: 1
INFO:distributed.scheduler:Starting worker compute stream, tcp://127.0.0.1:42763
INFO:distributed.core:Starting established connection to tcp://127.0.0.1:35612
INFO:distributed.scheduler:Register worker addr: tcp://127.0.0.1:35203 name: 0
INFO:distributed.scheduler:Starting worker compute stream, tcp://127.0.0.1:35203
INFO:distributed.core:Starting established connection to tcp://127

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:8787/status,
Dashboard: http://127.0.0.1:8787/status,Workers: 2
Total threads: 4,Total memory: 12.67 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:36901,Workers: 0
Dashboard: http://127.0.0.1:8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:35203,Total threads: 2
Dashboard: http://127.0.0.1:36785/status,Memory: 6.34 GiB
Nanny: tcp://127.0.0.1:33541,


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
cd /content/drive/MyDrive/etl

/content/drive/MyDrive/etl


In [ ]:
ls

austinHousingData.csv  dask-expr.svg  homeImages/  mydask.png


In [ ]:
# import libraries
import sys
import os

## import dask libraries
import dask.dataframe as dd
from dask.diagnostics import ProgressBar

# import libraries
import pandas as pd

In [ ]:
cwd = os.getcwd()

# print
print('', sys.executable)
print('', cwd)

 /usr/bin/python3
 /content/drive/MyDrive/etl


#### Шаг 1. Extract (Извлечение данных)

In [ ]:
df = dd.read_csv('austinHousingData.csv',  dtype={'zipcode': 'object', 'latest_saledate': 'object', 'description': 'object', 'homeImage': 'object'})
df

,zpid,city,streetAddress,zipcode,description,latitude,longitude,propertyTaxRate,garageSpaces,hasAssociation,hasCooling,hasGarage,hasHeating,hasSpa,hasView,homeType,parkingSpaces,yearBuilt,latestPrice,numPriceChanges,latest_saledate,latest_salemonth,latest_saleyear,latestPriceSource,numOfPhotos,numOfAccessibilityFeatures,numOfAppliances,numOfParkingFeatures,numOfPatioAndPorchFeatures,numOfSecurityFeatures,numOfWaterfrontFeatures,numOfWindowFeatures,numOfCommunityFeatures,lotSizeSqFt,livingAreaSqFt,numOfPrimarySchools,numOfElementarySchools,numOfMiddleSchools,numOfHighSchools,avgSchoolDistance,avgSchoolRating,avgSchoolSize,MedianStudentsPerTeacher,numOfBathrooms,numOfBedrooms,numOfStories,homeImage
npartitions=1,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
,int64,string,string,string,string,float64,float64,float64,int64,bool,bool,bool,bool,bool,bool,string,int64,int64,float64,int64,string,int64,int64,string,int64,int64,int64,int64,int64,int64,int64,int64,int64,float64,float64,int64,int64,int64,int64,float64,float64,int64,int64,float64,int64,int64,string
,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...


In [ ]:
images_folder = 'homeImages'

sample = df.head(5)
for _, row in sample.iterrows():
    photo_file = row['homeImage']
    if pd.notna(photo_file):
        full_path = os.path.join(images_folder, photo_file)
        if os.path.exists(full_path):
            print(f"{photo_file} - найдено")
        else:
            print(f"{photo_file} - нет в папке {images_folder}")

ERROR:asyncio:Task exception was never retrieved
future: <Task finished name='Task-77015' coro=<Client._gather.<locals>.wait() done, defined at /usr/local/lib/python3.12/dist-packages/distributed/client.py:2388> exception=AllExit()>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/distributed/client.py", line 2397, in wait
    raise AllExit()
distributed.client.AllExit


KeyboardInterrupt: 

#### Шаг 2. Transform (Трансформация и очистка данных)

In [ ]:
# count missing values
missing_values = df.isnull().sum()
missing_values

Dask Series Structure:
npartitions=1
MedianStudentsPerTeacher    int64
zpid                          ...
Dask Name: sum, 5 expressions
Expr=(~ NotNull(frame=ArrowStringConversion(frame=FromMapProjectable(be1eafc)))).sum()

In [ ]:
# calculate percent missing values
mysize = df.index.size
missing_count = ((missing_values / mysize) * 100)
missing_count

Dask Series Structure:
npartitions=1
MedianStudentsPerTeacher    float64
zpid                            ...
Dask Name: mul, 9 expressions
Expr=(~ NotNull(frame=ArrowStringConversion(frame=FromMapProjectable(be1eafc)))).sum() / Index(frame=ArrowStringConversion(frame=FromMapProjectable(be1eafc))).size() * 100

In [ ]:
# запуск вычисления, используя метод подсчета
with ProgressBar():
  missing_count_percent = missing_count.compute()
missing_count_percent

In [ ]:
available_images = set(os.listdir(images_folder))
print(f"Найдено фото в папке: {len(available_images)}")

# Функция для проверки существования фото
def has_photo(image_name):
    if pd.isna(image_name) or image_name == '':
        return False
    return image_name in available_images

df['photo_exists'] = df['homeImage'].map(
    has_photo,
    meta=('photo_exists', 'bool')
)

total_houses = len(df)

# Дома с фото
houses_with_photo = df['photo_exists'].sum().compute()

# Дома без фото
houses_without_photo = total_houses - houses_with_photo

# Дома с пустым значением homeImage
empty_homeimage = df['homeImage'].isna().sum().compute()
empty_homeimage += (df['homeImage'] == '').sum().compute()

print(f"Домов С фото: {houses_with_photo:,} ({houses_with_photo/total_houses*100:.2f}%)")
print(f"Домов без фото: {houses_without_photo:,} ({houses_without_photo/total_houses*100:.2f}%)")
print(f"Домов с пустым homeImage: {empty_homeimage:,}")

#### Шаг 3. Load (Сохранение результатов пайплайна)